## 0. 세팅

In [ ]:
# API 키 세팅
import os
from config.config import config
os.environ["OPENAI_API_KEY"] = config["apikey"]["openai"]

# 경고 무시 세팅
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

## 1. RAG 준비 - 청킹 및 벡터스토어

### (1) 파일 체크  

1. 사용할 파일 : 뉴욕 2050 전략 계획, 서울 2040 계획

In [ ]:
from glob import glob 

for g in glob('data/*.pdf'):
    print(g)

### (2) 청크 단위로 텍스트를 잘라 반환하는 함수

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """
    주어진 PDF 파일을 읽고 텍스트를 분할합니다.
    매개변수:
        pdf_path (str): PDF 파일의 경로.
        chunk_size (int, 선택적): 각 텍스트 청크의 크기. 기본값은 1000입니다.
        chunk_overlap (int, 선택적): 청크 간의 중첩 크기. 기본값은 100입니다.
    반환값:
        list: 분할된 텍스트 청크의 리스트.
    """
    print(f"PDF: {pdf_path} -----------------------------")

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(data_from_pdf)
    
    print(f"Number of splits: {len(splits)}\n")
    return splits


### (3) 문서를 청킹해 Chroma DB에 적재  

- chroma_store 디렉터리가 없는 상태라면 새로이 청킹 및 임베딩하여 적재함

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

##### Vectorstore 설정 #####
embedding = OpenAIEmbeddings(model='text-embedding-3-large')

persist_directory='chroma_store'

if os.path.exists(persist_directory):
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory, 
        embedding_function=embedding
    )
else:
    print("Creating new Chroma store")
    
    vectorstore = None
    for g in glob('data/*.pdf'):
        chunks = read_pdf_and_split_text(g)
        # 100개씩 나눠서 저장
        for i in range(0, len(chunks), 100):
            if vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=chunks[i:i+100],
                    embedding=embedding,
                    persist_directory=persist_directory
                )
            else:
                vectorstore.add_documents(
                    documents=chunks[i:i+100]
                )


### (4) 벡터 DB 동작 확인

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

chunks = retriever.invoke("서울 온실가스 저감 계획")

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)


## 2. 라우터

### 개념  

- 입력한 내용에 따라 여러 개의 실행 경로 중, 적절한 경로를 결정해 다음 노드를 선택하는 기능  
- 특정 조건에 따라 다르게 동작해야 할 때 사용함  
- 이번 실습에서는 사용자의 발화 의도를 이분법으로 분석하는 "의도분류자" 라우터를 구성한다.  

### (1) 사용할 챗봇 정의

In [ ]:
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")
model.invoke('안녕하세요!')

### (2) 라우터 설정하기  

- LLM 에게 사용자의 쿼리(발화)가 어떠한 의도를 가졌는지 판별을 요청  
- 이 답변에 따라 후속 작동 방식을 정의하여 라우터를 구성한다.  

In [ ]:
# (1) Router 설정

from langchain_core.prompts import ChatPromptTemplate
from typing import Literal # 문자열 리터럴 타입을 지원하는 typing 모듈의 클래스
from pydantic import BaseModel, Field

# Data model
class RouteQuery(BaseModel):
    """사용자 쿼리를 가장 관련성이 높은 데이터 소스로 라우팅합니다."""
    
    datasource: Literal["vectorstore", "casual_talk"] = Field(
        ...,
        description="""
        사용자 질문에 따라 casual_talk 또는 vectorstore로 라우팅합니다.
        - casual_talk: 일상 대화를 위한 데이터 소스. 사용자가 일상적인 질문을 할 때 사용합니다.
        - vectorstore: 사용자 질문에 답하기 위해 RAG로 vectorstore 검색이 필요한 경우 사용합니다.
        """,
    )

In [ ]:
# (2) 랭체인 기능을 이용해 라우터 정의

# 특정 모델을 structured output (구조화된 출력)과 함께 사용하기 위해 설정
structured_llm_router = model.with_structured_output(RouteQuery)

router_system = """
당신은 사용자의 질문을 vectorstore 또는 casual_talk으로 라우팅하는 전문가입니다.
- vectorstore에는 서울, 뉴욕의 발전계획과 관련된 문서가 포함되어 있습니다. 이 주제에 대한 질문에는 vectorstore를 사용하십시오.
- 사용자의 질문이 일상 대화에 관련된 경우 casual_talk을 사용하십시오.
"""

# 시스템 메시지와 사용자의 질문을 포함하는 프롬프트 템플릿 생성
route_prompt = ChatPromptTemplate.from_messages([
    ("system", router_system),
    ("human", "{question}"),
])

# 라우터 프롬프트와 구조화된 출력 모델을 결합한 객체
question_router = route_prompt | structured_llm_router


In [ ]:
# (3) 테스트 : 

# 1) RAG로 vectorstore 검색이 필요한 질의
print( "1. RAG 의도 : ", 
    question_router.invoke({
        "question": "서울 온실가스 저감 계획은 무엇인가요?"
    })
)

# 2) 일상적인 대화
print("2. 일상 의도 : ", question_router.invoke({"question": "잘 지냈어?"}))

## 3. 랭그래프로 RAG 에이전트 만들기 -> 랭그래프가 아니라 그냥 RAG 에이전트 만들기

### (1) Vector 검색된 문서의 연관성 검증 / 필터링

In [ ]:
# 검색 문서 연관성 검증 에이전트

from langchain_core.prompts import PromptTemplate

class GradeDocuments(BaseModel):
    """검색된 문서가 질문과 관련성 있는지 yes 또는 no로 평가합니다."""

    binary_score: Literal["yes", "no"] = Field(
        description="문서가 질문과 관련이 있는지 여부를 'yes' 또는 'no'로 평가합니다."
    )

# (1) 검증 모델
structured_llm_grader = model.with_structured_output(GradeDocuments)

# (2) 검증 프롬프트
grader_prompt = PromptTemplate.from_template("""
당신은 검색된 문서가 사용자 질문과 관련이 있는지 평가하는 평가자입니다. \n 
문서에 사용자 질문과 관련된 키워드 또는 의미가 포함되어 있으면, 해당 문서를 관련성이 있다고 평가하십시오. \n
엄격한 테스트가 필요하지 않습니다. 목표는 잘못된 검색 결과를 걸러내는 것입니다. \n
문서가 질문과 관련이 있는지 여부를 나타내기 위해 'yes' 또는 'no'로 이진 점수를 부여하십시오.
                                             
Retrieved document: \n {document} \n\n 
User question: {question}
""")

# (3) 체인 정의
retrieval_grader = grader_prompt | structured_llm_grader

In [ ]:
# 검색
question = "서울시 자율주행 관련 계획"
documents = retriever.invoke(question)

print("=== 검색 결과 ===")
for doc in documents:
    print(doc)


In [ ]:
# 검색된 문서들에 대한 검증

filtered_docs = []

for i, doc in enumerate(documents):
    print(f"Document {i + 1}:")
    is_relevant = retrieval_grader.invoke({"question": question, "document": doc.page_content})
    print(is_relevant)
    print(doc.page_content[:200])
    print("=================================\n\n")

    if is_relevant.binary_score == "yes":
        filtered_docs.append(doc)

print(f"Filtered documents: {len(filtered_docs)}")


### (2) RAG 답변 생성하기

In [ ]:
### Generate
# PromptTemplate을 사용하여 RAG를 위한 프롬프트를 생성합니다.

rag_generate_system = """
너는 사용자의 질문에 대해 주어진 context에 기반하여 답변하는 도시 계획 전문가이다. 
주어진 context는 vectorstore에서 검색된 결과이다. 
주어진 context를 기반으로 사용자의 question에 대해 답변하라.

=================================
question: {question}
context: {context}
"""

# PromptTemplate을 생성하여 question과 context를 포맷팅
rag_prompt = PromptTemplate(
    input_variables=["question", "context"],
    template=rag_generate_system
)

# rag chain
rag_chain = rag_prompt | model 

# 사용자 질문과 검색된 문서를 입력으로 사용하여 RAG를 실행
question = "서울시 자율주행 관련 계획"

rag_chain.invoke({"question": question, "context": filtered_docs})


"""답변
서울시의 자율주행 관련 계획은 미래교통 기술의 발전에 대응하기 위한 포괄적인 전략을 포함하고 있습니다.
서울시는 2030년까지 간선도로급 이상의 도로에서 자율주행 자동차가 운영될 수 있는 인프라 환경을 조성할 계획이며,
2040년까지는 서울 전역에서 자율주행차의 운행 환경을 구축하고 수송 분담률 10%를 달성한다는 목표를 설정하고 있습니다.
또한, 서울시는 도심 항공교통(UAM) 기반을 마련하고, 주요 수변 공간에 광역노선을 단계적으로 확장하는 한편,
서울 전역에 모빌리티 허브를 구축하여 기존 교통과 미래 교통을 연결하는 다양한 기능이 복합적으로 제공되는 지역 거점을 마련할 계획입니다.
이러한 접근은 안전하고 효율적인 교통 환경을 조성하고,
새로운 교통수단의 정착을 위한 가이드라인 마련을 포함하여 시가지 공간 변화에 대한 포괄적인 대응책을 마련하는 것을 목표로 하고 있습니다.
"""


## 4. 그래프 정의하기  

### (1) 그래프의 구성 요소  

1. 상태 : 그래프 전체에 있어서 공통적으로 사용하는 데이터, 데이터의 흐름을 보관하는 개체  
2. 노드 : 특정한 작업을 실제로 수행하는 작업자  
3. 엣지 : 하나의 노드와 하나 이상의 노드를 잇는 작업의 연결경로이자 순서  

### (2) 상태(State) 선언

In [ ]:
from typing import List
from typing_extensions import TypedDict

class GraphState(TypedDict):
    question: str   # 사용자 질문
    generation: str # LLM 생성 결과
    documents: List[str] # 검색된 문서


### (3) 질문 의도를 분류하는 라우터 함수 정의

In [ ]:
def route_question(state): 
    """
    사용자 질문을 vectorstore 또는 casual_talk로 라우팅합니다.
    
    Args:
        state (dict): 현재 graph state

    return:
        state (dict): 라우팅된 데이터 소스와 사용자 질문을 포함하는 새로운 graph state
    """
    print('------ROUTE------')
    question = state['question']
    route = question_router.invoke({"question": question})

    
    print(f"---Routing to {route.datasource}---")
    return route.datasource   

### (4) 검색 노드 정의

In [ ]:
def retrieve(state): 
    """
    vectorstore에서 질문에 대한 문서를 검색합니다.
    
    Args:
        state (dict): 현재 graph state

    return:
        state (dict): 검색된 문서와 사용자 질문을 포함하는 새로운 graph state
    """
    print('------RETRIEVE------')
    question = state['question']

    # Retrieve documents
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}

### (5) 관련도 평가 및 필터링 노드 정의

In [ ]:
def grade_documents(state):
    """
    검색된 문서를 평가하여 질문과 관련성이 있는지 확인합니다.

    Args:
        state (dict): 현재 graph state

    return:
        state (dict): 관련성이 있는 문서와 사용자 질문을 포함하는 새로운 graph state
    """
    print('------GRADE------')
    question = state['question']
    documents = state['documents']
    filtered_docs = []

    for i, doc in enumerate(documents):
        is_relevant = retrieval_grader.invoke({"question": question, "document": doc.page_content})
        if is_relevant.binary_score == "yes":
            filtered_docs.append(doc)
    return {"documents": filtered_docs, "question": question}  

### (6) LLM 답변 생성 노드 정의

In [ ]:
# 1) RAG에 따른 답변을 수행하는 노드

def generate(state):
    """
    LLM을 사용하여 문서와 사용자 질문에 대한 답변을 생성합니다.

    Args:
        state (dict): 현재 graph state

    return:
        state (dict): LLM 생성 결과와 사용자 질문을 포함하는 새로운 graph state
    """
    print('------GENERATE------')
    question = state['question']
    documents = state['documents']
    generation = rag_chain.invoke({"question": question, "context": documents})
    return {
        "documents": documents,
        "question": question,
        "generation": generation
    }

In [ ]:
# 2) 일상대화에 대한 답변을 수행하는 노드

def casual_talk(state):
    """
    일상 대화를 위한 답변을 생성합니다.

    Args:
        state (dict): 현재 graph state

    return:
        state (dict): 일상 대화 결과와 사용자 질문을 포함하는 새로운 graph state
    """
    print('------CASUAL TALK------')
    question = state['question']
    generation = model.invoke(question)
    return {
        "question": question,
        "generation": generation
    }

### (7) StateGraph 정의

In [ ]:
from langgraph.graph import START, StateGraph, END

workflow = StateGraph(GraphState)

In [ ]:
# 노드를 정의 
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("casual_talk", casual_talk)

In [ ]:
# graph를 정의
workflow.add_conditional_edges(        # Graph의 앞단 : START 노드 -> 의도분류 라우터
    START, 
    route_question,
    {
        "vectorstore": "retrieve",
        "casual_talk": "casual_talk"
    }
)
workflow.add_edge("casual_talk", END)  # 일상대화 노드 -> 종료
workflow.add_edge("retrieve", "grade_documents") # 검색노드 -> 관련성 평가 및 필터 노드
workflow.add_edge("grade_documents", "generate") # 관련성 평가 및 필터 노드 -> LLM 답변 생성 노드
workflow.add_edge("generate", END) # LLM 답변 생성 노드 -> END 노드

app = workflow.compile() # workflow를 컴파일

In [ ]:
# 정의한, 컴파일된 랭그래프를 도식화

from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    #  실패 시 통과
    pass


### (8) 멀티에이전트 테스트

In [ ]:
# 1) RAG 가 필요한 경우

inputs = {
    "question": "서울시 자율주행 계획"
}

result = app.invoke(inputs) # workflow를 실행합니다.
print(result)

"""답변
서울시의 자율주행 계획은 단계적으로 발전하고 있으며, 2030년까지 간선도로급 이상의 도로에서 자율주행 자동차 운영이 가능하도록 인프라를 구축하는 것이 목표입니다.
2040년까지는 서울 전역에서 자율주행 차량이 운행될 수 있는 환경을 완비하고, 이를 통해 교통 수송의 분담률 10%를 달성할 계획입니다.
이와 함께 서울시는 도심 항공교통(UAM)을 위한 기반을 마련하고, 시범노선을 운영하는 등 상용화 방향으로 나아가고 있습니다.
주요 수변 공간에 광역 노선을 확장하는 인프라 구축도 검토되고 있습니다.
또한, 서울 전역에 모빌리티 허브를 설계해 기존 교통과 미래 교통을 효과적으로 연결하는 다양한 기능을 통합적으로 제공할 예정입니다.
이를 통해 안전하고 효율적인 교통체계로의 전환을 목표로 하고 있으며,
교통사고를 줄이고, 새로운 교통 환경에 맞는 정착 가이드라인 마련도 필요하다고 강조하고 있습니다.
"""

In [ ]:
# 2) 일상 대화

inputs = {
    "question": "잘 지내고 있어?"
}

result = app.invoke(inputs) # workflow를 실행합니다.
print(result)

"""답변
네, 잘 지내고 있습니다! 당신은 어떻게 지내고 계신가요?
"""

In [ ]:
# 3) 스트림 방식으로 출력

inputs = {
    "question": "서울시의 자율주행 차량 관련 계획은 무엇이 있나요?"
}

for msg, meta in app.stream(inputs, stream_mode='messages'):
    print(msg.content, end='')
